# Chapter 17: Investment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quarcs-lab/macro102-site/blob/main/notebooks/ch17-investment.ipynb)

*Data Science and Macroeconomic Models: A Python Companion to Intermediate Macroeconomics* · Carlos Mendez

> **How to use this notebook.** In Colab, click *Runtime → Run all*. After that, every model
> figure has live sliders: move one and the model is solved again on the spot. On the book
> website the same figures come with one precomputed slider. Each section ends with a short
> **Your turn** task, and you can try it in the empty code cell below the task.

## Learning goals

Investment links the present to the future: a firm that buys a machine today gives up money now
for output later. One tool, the **arbitrage equation**, explains investment in machines, in stocks
and in houses. By the end of this chapter you will be able to:

- use an arbitrage equation to compare two ways of investing the same money;
- compute the **user cost of capital** and find the **desired capital stock**, where the marginal
  product of capital equals the user cost;
- show how a **corporate income tax** raises the user cost and lowers capital and the investment rate,
  and rebuild Table 17.1 for any year;
- price a stock as the present value of its dividends, $\text{dividend}/(R-g)$, and use this formula
  to read the **price-earnings ratio** of the stock market;
- describe the components of U.S. private investment and how they move in recessions;
- write Python functions with default values, draw Plotly figures with sliders and buttons, reshape
  tables with `rename`, `melt` and `merge`, and repeat steps with `for` and `while` loops.

## Figures in this chapter

The figures follow the order of the textbook chapter. The second column shows which textbook
figure each notebook figure replicates.

| Figure | Textbook figure | Type | What you can change |
|---|---|---|---|
| Figure 1 | 17.1 How much should a firm invest? | model | interest rate, depreciation rate, capital gain, productivity, starting capital |
| Figure 2 | 17.2 An increase in the corporate income tax | model | tax rate, interest rate, depreciation rate, "Introduce the tax!" button; Table 17.1 for any year |
| Figure 3 | 17.3 The price-earnings ratio in the stock market | data | interest rate, dividend growth and payout ratio of the model line |
| Figure 4 | 17.4 Components of private investment (as a share of GDP) | data | annual or quarterly data, components on/off in the legend |

## Setup

We first load the Python libraries we need. You only have to run this cell once.

> **Python note.** `import` loads a *library*, a toolbox written by other people. We give each
> library a short nickname, for example `np` for NumPy, so that the code stays short. Text after
> a `#` is a *comment*: Python ignores it, and it is there for human readers. The `try ... except`
> block runs one Colab-only line and simply skips it anywhere else.

In [1]:
# Libraries used in this chapter
import numpy as np                              # numbers and arrays
import pandas as pd                             # data tables
import plotly.express as px                     # quick charts from data tables
import plotly.graph_objects as go               # charts built piece by piece (model figures)
import plotly.io as pio
from ipywidgets import interact, interact_manual, FloatSlider, ToggleButtons, Dropdown

# One light blue/green look for every chart
BLUE, TEAL, GREEN, GRAY = "#16528c", "#008070", "#7fb23a", "#8a96a3"
pio.templates.default = "plotly_white"
pio.templates["plotly_white"].layout.update(width=860, height=480)   # same size in Colab and on the website
px.defaults.color_discrete_sequence = [BLUE, TEAL, GREEN, "#5aa0d8", "#0b4f47"]
WIDE = {"description_width": "initial"}         # show the full slider labels

# Colab only: let Plotly charts appear inside the interactive widgets
try:
    import google.colab
    google.colab.output.enable_custom_widget_manager()
except ImportError:
    pass                                        # not running in Colab: nothing to do

# The data files of this chapter live in the book's GitHub repository
DATA_URL = "https://raw.githubusercontent.com/quarcs-lab/macro102-site/main/data/ch17/"

The next cell defines a helper that only the book website needs. Run it and move on.

In [2]:
# @title Helper for the website versions (run this cell; you do not need to read it)
def slider_figure(make_figure, name, values, start, label=None):
    """Precompute make_figure(name=value) for every value and join the results with one slider.

    The website cannot run Python, so each interactive figure also gets this precomputed version.
    """
    figures = [make_figure(**{name: v}) for v in values]
    first = values.index(start)
    fig = go.Figure(layout=figures[first].layout)
    owner = []                                  # which precomputed figure each trace comes from
    for i, f in enumerate(figures):
        for trace in f.data:
            trace.visible = (i == first)
            fig.add_trace(trace)
            owner.append(i)
    steps = []
    for i, f in enumerate(figures):
        layout = {"title": f.layout.title.to_plotly_json(),
                  "annotations": [a.to_plotly_json() for a in f.layout.annotations],
                  "shapes": [s.to_plotly_json() for s in f.layout.shapes]}
        visible = [o == i for o in owner]
        steps.append(dict(method="update", label=str(values[i]), args=[{"visible": visible}, layout]))
    fig.update_layout(sliders=[dict(active=first, steps=steps, pad=dict(t=60),
                                    currentvalue=dict(prefix=(label or name) + " = "))])
    return fig

## The model in code

**The arbitrage equation.** Gino's pizza restaurant has some cash. It can put the money in the bank
and earn the interest rate $R$, or it can buy a pizza oven at the price $p_k$, use it for a year
and sell it. If Gino's maximizes profits, the last dollar must earn the same in both uses, otherwise
moving money to the better use would raise profits:

$$\underbrace{R\cdot p_k}_{\text{bank account}} = \underbrace{MPK + \Delta p_k}_{\text{pizza oven}} \qquad (17.2)$$

The oven earns its **marginal product** $MPK$, plus a **capital gain** $\Delta p_k$ when its price
rises (a capital loss when it falls). With the price normalized to 1 and depreciation $\bar{d}$
written out explicitly, the firm invests until

$$MPK = \underbrace{R + \bar{d} - \frac{\Delta p_k}{p_k}}_{\text{user cost of capital}} \qquad (17.4)$$

The **user cost of capital** is the full cost of using one more unit of capital for a year: the
interest given up, the wear and tear, minus any rise in the price of the capital good. A corporate
income tax at rate $\tau$ takes part of the marginal product, so the marginal product must be higher:

$$MPK = \frac{R + \bar{d} - \Delta p_k/p_k}{1-\tau} \qquad (17.6)$$

**Diminishing returns.** With the production function of Chapter 4, $Y = \bar{A}K^{1/3}L^{2/3}$, the
marginal product of capital falls as capital rises:

$$MPK = \frac{1}{3}\cdot\frac{Y}{K} = \frac{1}{3}\cdot\bar{A}\left(\frac{L}{K}\right)^{2/3} \qquad (17.7)$$

(The textbook's text prints $L^{1/2}$ here, a typo: equation 17.7 needs the exponent $2/3$.)
Setting $MPK$ equal to the user cost $uc$ gives the **desired capital stock**,
$K^{\text{desired}} = L\,(\bar{A}/(3\,uc))^{3/2}$, and the capital-output ratio $Y/K = 3\cdot uc$ (17.9).

**Our baseline.** The textbook's own example has a 2 percent real interest rate, an 8 percent
depreciation rate and no capital gain, so the user cost is 10 percent. We add $\bar{A} = 0.3$ and
$L = 1$, chosen so that the desired capital stock is exactly $K = 1$ (and output $Y = 0.3$). Figures
1 and 2 both start from this baseline.

> **Python note.** `def` creates a *function*: a named recipe that takes inputs (the
> *arguments*) and gives back a result with `return`. Writing `gpk=0.0` gives the argument a
> *default value*, used whenever we do not pass one. The text in triple quotes is a short
> description of what the function does. `**` means "to the power of". Names in capital letters,
> such as `A_BAR`, are constants that we set once and reuse.

In [3]:
A_BAR, L_BAR = 0.3, 1.0          # illustrative productivity and labor: the baseline desired capital is K = 1


def user_cost(R, d, gpk=0.0, tau=0.0):
    """User cost of capital: (R + d - gpk) / (1 - tau)   (equations 17.4 and 17.6; gpk is the capital gain)."""
    return (R + d - gpk) / (1 - tau)


def mpk(K, A=A_BAR, L=L_BAR):
    """Marginal product of capital: MPK = (1/3) A (L/K)^(2/3)   (equation 17.7)."""
    return A / 3 * (L / K) ** (2 / 3)


def desired_capital(uc, A=A_BAR, L=L_BAR):
    """Desired capital stock, where MPK = uc: K = L (A / (3 uc))^(3/2)   (from equation 17.7)."""
    return L * (A / (3 * uc)) ** (3 / 2)


def investment_rate(uc, g_K=0.02, d=0.08):
    """Investment rate: I/Y = (g_K + d) / (3 uc)   (equation 17.13)."""
    return (g_K + d) / (3 * uc)

The same arbitrage idea prices a stock. An investor can earn $R$ in the bank, or buy a stock at the
price $p_s$ and receive its dividend plus the capital gain. The two returns must be equal:

$$R = \frac{\text{dividend}}{p_s} + \frac{\Delta p_s}{p_s} \qquad (17.15)$$

When dividends grow at a constant rate $g$, the stock price grows at $g$ too, and solving for the
price gives

$$p_s = \frac{\text{dividend}}{R - g} \qquad (17.16)$$

Dividing by the earnings of the firm gives the price-earnings ratio, which depends on the share of
earnings paid out as dividends (the **payout ratio**) and on $R-g$:

$$\frac{p_s}{\text{earnings}} = \frac{\text{dividend}/\text{earnings}}{R - g} \qquad (17.17)$$

In [4]:
def stock_price(dividend, R, g=0.0):
    """Price of a stock: dividend / (R - g)   (equation 17.16; dividends grow at rate g)."""
    return dividend / (R - g)


def pe_ratio(payout, R, g):
    """Price-earnings ratio: (dividend / earnings) / (R - g)   (equation 17.17)."""
    return payout / (R - g)

Let us check the functions against the numbers in the textbook.

> **Python note.** An *f-string* such as `f"{uc:.1%}"` inserts the value of a variable into a
> piece of text; `:.1%` shows a share as a percentage with one decimal (0.1 becomes 10.0%), and
> `:.2f` shows two decimals.

In [5]:
uc = user_cost(R=0.02, d=0.08)
uc_tax = user_cost(R=0.02, d=0.08, tau=0.35)
K = desired_capital(uc)
print(f"User cost, no tax:            {uc:.1%}   (textbook: 10 percent)")
print(f"User cost with a 35% tax:     {uc_tax:.1%}   (textbook: 'more than 15 percent')")
print(f"Desired capital, baseline:    K = {K:.2f}, with MPK = {mpk(K):.1%}")
print(f"Capital-output ratio check:   Y/K = {A_BAR * K ** (1 / 3) / K:.2f} = 3 x uc = {3 * uc:.2f}   (17.9)")
print(f"Stock, 10 dollar dividend, R = 5%:          {stock_price(10, 0.05):.0f} dollars   (textbook: 200)")
print(f"Stock, 10 dollar dividend, R = 5%, g = 2%:  {stock_price(10, 0.05, 0.02):.0f} dollars   (textbook: about 333)")

User cost, no tax:            10.0%   (textbook: 10 percent)
User cost with a 35% tax:     15.4%   (textbook: 'more than 15 percent')
Desired capital, baseline:    K = 1.00, with MPK = 10.0%
Capital-output ratio check:   Y/K = 0.30 = 3 x uc = 0.30   (17.9)
Stock, 10 dollar dividend, R = 5%:          200 dollars   (textbook: 200)
Stock, 10 dollar dividend, R = 5%, g = 2%:  333 dollars   (textbook: about 333)


The exact user cost with the tax is $0.10/0.65 = 0.1538$, which rounds to 15.4 percent; the
textbook writes $\approx 0.153$.

### Drawing tools

Figures 1 and 2 are the same kind of diagram, so we write its drawing steps once:

- `capital_figure` starts a diagram with fixed axes, so that you can see the curves move;
- `add_mpk` draws the marginal product curve, and `add_level` draws a flat user-cost line; both
  write the name of the line on the plot, as the textbook does, and `add_label` skips a name
  that would sit on top of another one;
- `add_point` marks a desired capital stock with a dashed guide to the axis, and `add_arrow` draws an arrow.

> **Python note.** `np.linspace(0.12, 3, 300)` creates 300 evenly spaced numbers from 0.12 to 3.
> Because `mpk` works on a whole array at once, one call gives the whole curve. `go.Figure()`
> starts an empty chart, and each `add_trace` adds one line or set of points. `MPK <= 0.4` is
> true where the curve is inside the axes, `K[inside]` keeps those values, and `[0]` takes the
> first of them. A `for` loop over `fig.layout.annotations` looks at every label already written,
> and `return` inside the loop stops the function early.

In [6]:
K_RANGE, RETURN_RANGE = [0, 3], [0, 0.4]          # fixed axes of Figures 1 and 2


def capital_figure(title):
    """An empty diagram: capital on the x-axis, returns per unit of capital (percent) on the y-axis."""
    fig = go.Figure()
    fig.update_layout(title=title, showlegend=False,
                      xaxis=dict(title="Capital, K", range=K_RANGE, tickvals=[0.5, 1, 1.5, 2, 2.5, 3], zeroline=False),
                      yaxis=dict(title="Return per unit of capital", range=RETURN_RANGE,
                                 tickvals=[0, 0.1, 0.2, 0.3, 0.4], tickformat=".0%", zeroline=False))
    return fig


def add_label(fig, x, y, text, color, xanchor="left"):
    """Write a line name at (x, y), unless another name already sits within 2 points of it."""
    for old in fig.layout.annotations:
        if old.showarrow is False and abs(old.x - x) < 0.5 and abs(old.y - y) < 0.02:
            return
    fig.add_annotation(x=x, y=y, text=text, showarrow=False, xanchor=xanchor, yanchor="bottom",
                       font=dict(color=color, size=14))

In [7]:
def add_mpk(fig, A=A_BAR):
    """Draw the MPK curve of equation (17.7) and write its name next to its top-left end."""
    K = np.linspace(0.12, K_RANGE[1], 300)
    MPK = mpk(K, A)
    inside = MPK <= RETURN_RANGE[1]                       # the part of the curve inside the axes
    fig.add_trace(go.Scatter(x=K[inside], y=MPK[inside], mode="lines", name="MPK", line=dict(color=BLUE, width=3)))
    add_label(fig, K[inside][0] + 0.05, MPK[inside][0] - 0.03, "Marginal product of capital, MPK", BLUE)


def add_level(fig, uc, label, color, dash="solid"):
    """Draw a flat user-cost line at the height uc and write its name at its right end."""
    fig.add_trace(go.Scatter(x=K_RANGE, y=[uc, uc], mode="lines", name=label, line=dict(color=color, width=3, dash=dash)))
    add_label(fig, K_RANGE[1], uc, label, color, xanchor="right")


def add_point(fig, K, uc, label, anchor="left"):
    """Mark (K, uc) with a dot and a dashed guide down to the x-axis, labelled next to the guide."""
    fig.add_trace(go.Scatter(x=[K, K], y=[0, uc], mode="lines", hoverinfo="skip",
                             line=dict(color="black", width=1, dash="dash")))
    fig.add_trace(go.Scatter(x=[K], y=[uc], mode="markers", name=label, marker=dict(color="black", size=11)))
    fig.add_annotation(x=K, y=0, text=label, showarrow=False, xanchor=anchor, yanchor="bottom", font=dict(size=14))


def add_arrow(fig, x0, y0, x1, y1, text=""):
    """Draw an arrow from (x0, y0) to (x1, y1), with an optional text at its tail; nothing if the points almost coincide."""
    if abs(x1 - x0) > 0.03 or abs(y1 - y0) > 0.005:
        fig.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref="x", yref="y", axref="x", ayref="y", text=text,
                           showarrow=True, arrowhead=3, arrowwidth=2, arrowcolor="black")

## Figure 1: How much should a firm invest?

**Goal.** Find the desired capital stock of a firm, and see how the interest rate, depreciation,
capital gains and productivity change it.

The marginal product of capital falls as the firm adds capital: the first pizza oven adds a lot of
pizzas, the tenth adds few. The user cost does not depend on how much capital the firm has, so it is
a flat line. The firm keeps investing as long as one more unit of capital earns more than it costs,
so it stops where the two lines cross, at $K^{\text{desired}}$.

Investment is what moves the firm toward that point. The capital accumulation equation says that
next year's capital is this year's investment plus the capital that did not wear out:

$$K_{t+1} = I_t + (1-\bar{d})\,K_t \qquad (17.8)$$

So a young firm that starts with $K_0$ and wants $K^{\text{desired}}$ next year must invest
$I = K^{\text{desired}} - (1-\bar{d})K_0$. The title shows this number. The arrow points from $K_0$ to
the desired capital stock.

In [8]:
def desired_capital_diagram(R=0.02, d=0.08, gpk=0.0, A=A_BAR, K0=0.5):
    """Figure 17.1: invest until the marginal product of capital falls to the user cost."""
    uc = user_cost(R, d, gpk)
    K_star = desired_capital(uc, A)
    I = K_star - (1 - d) * K0                        # investment that reaches K_star next year, equation (17.8)
    fig = capital_figure(f"User cost {uc:.1%}  |  desired capital {K_star:.2f}  |  investment from K0 = {K0:.1f}: {I:+.2f}")
    add_mpk(fig, A)
    add_level(fig, uc, "User cost: R + d − Δp/p", TEAL)
    add_point(fig, K_star, uc, "K<sup>desired</sup>")
    add_arrow(fig, K0, 0.03, K_star, 0.03, text="K<sub>0</sub>")
    return fig

Now the interactive version. Move the sliders and watch where the two lines cross.

> **Python note.** `interact` turns each argument of a function into a slider and calls the
> function again every time you move one. The `;` at the end only hides an extra line of text.

In [9]:
interact(desired_capital_diagram,
         R=FloatSlider(value=0.02, min=0.01, max=0.08, step=0.01, description="interest rate R", style=WIDE),
         d=FloatSlider(value=0.08, min=0.05, max=0.15, step=0.01, description="depreciation rate d", style=WIDE),
         gpk=FloatSlider(value=0.0, min=-0.04, max=0.04, step=0.01, description="capital gain Δp/p", style=WIDE),
         A=FloatSlider(value=0.3, min=0.15, max=0.45, step=0.05, description="productivity A", style=WIDE),
         K0=FloatSlider(value=0.5, min=0.1, max=2.5, step=0.1, description="starting capital K0", style=WIDE));

In [10]:
# @title Website version: Figure 1 with a precomputed slider for the interest rate
slider_figure(desired_capital_diagram, "R", [0.01, 0.02, 0.03, 0.04, 0.05, 0.06], start=0.02,
              label="interest rate R")

**What to notice.**

- In the baseline the user cost is 10 percent and the desired capital stock is 1. A firm that starts
  with $K_0 = 0.5$ must invest 0.54: half a unit to grow, and 0.04 to replace the capital that wears out.
- A higher interest rate or faster depreciation raises the user cost and moves $K^{\text{desired}}$ to
  the left. Because the MPK curve flattens out, a rise of 1 point in the user cost lowers desired
  capital by more when the user cost is low.
- A **capital gain** lowers the user cost. Structures, whose land gets scarcer, often gain in value;
  computers lose value quickly, so their user cost is high. Set $\Delta p/p = -0.04$: desired capital
  falls to 0.60.
- Higher productivity $\bar{A}$ shifts the MPK curve up, so the firm wants more capital at the same user cost.

**Your turn.** A firm can buy either computers, whose price falls by 4 percent a year, or a building,
whose price rises by 2 percent a year. With $R = 0.02$ and $\bar{d} = 0.08$ for both, how much more
capital does the firm want in buildings than in computers? Use `user_cost` and `desired_capital` in the
cell below.

<details><summary>Hint</summary>

The capital gain enters the user cost with a minus sign: `user_cost(0.02, 0.08, gpk=-0.04)` for the
computers and `gpk=0.02` for the building. Then divide the two desired capital stocks.

</details>

In [11]:
# Your turn: try it here

## Figure 2: An increase in the corporate income tax

**Goal.** See why a corporate income tax lowers the capital stock that firms want.

The tax takes a share $\tau$ of what each unit of capital earns, so the firm keeps only
$(1-\tau)MPK$. To make an investment worthwhile, the marginal product before tax must now cover the
user cost divided by $1-\tau$ (equation 17.6). The user-cost line shifts up, and the firm moves up
its MPK curve to a smaller capital stock. With the textbook's numbers, a 35 percent tax raises the
user cost from 10 to about 15 percent.

In [12]:
def corporate_tax_diagram(tau=0.35, R=0.02, d=0.08):
    """Figure 17.2: a corporate income tax raises the user cost and lowers the desired capital stock."""
    uc0, uc1 = user_cost(R, d), user_cost(R, d, tau=tau)
    K0, K1 = desired_capital(uc0), desired_capital(uc1)
    fig = capital_figure(f"User cost {uc0:.1%} → {uc1:.1%}  |  desired capital {K0:.2f} → {K1:.2f} ({K1 / K0 - 1:+.0%})")
    add_mpk(fig)
    add_level(fig, uc1, "User cost (with a tax)", TEAL)
    add_level(fig, uc0, "User cost (no tax)", GRAY, dash="dash")      # the old line, dashed
    add_point(fig, K0, uc0, "K<sup>initial</sup>")
    add_point(fig, K1, uc1, "K<sup>desired</sup>", anchor="right")
    add_arrow(fig, 0.06, uc0, 0.06, uc1)                              # the user cost shifts up
    add_arrow(fig, K0, 0.06, K1, 0.06)                                # desired capital falls
    return fig

The economy starts with no tax. Choose a tax rate, then press **Introduce the tax!**

> **Python note.** `interact_manual` works like `interact`, but adds a button: nothing is redrawn until
> you press it. `.options(manual_name=...)` sets the text on the button.

In [13]:
introduce = interact_manual.options(manual_name="Introduce the tax!")
introduce(corporate_tax_diagram,
          tau=FloatSlider(value=0.35, min=0.0, max=0.6, step=0.05, description="corporate tax rate τ", style=WIDE),
          R=FloatSlider(value=0.02, min=0.01, max=0.08, step=0.01, description="interest rate R", style=WIDE),
          d=FloatSlider(value=0.08, min=0.05, max=0.15, step=0.01, description="depreciation rate d", style=WIDE));

In [14]:
# @title Website version: Figure 2 with a precomputed slider for the corporate tax rate
slider_figure(corporate_tax_diagram, "tau", [0.0, 0.1, 0.2, 0.21, 0.3, 0.35, 0.4, 0.5], start=0.35,
              label="corporate tax rate τ")

**What to notice.**

- With $\tau = 0.35$ the user cost rises from 10.0 to 15.4 percent, and the desired capital stock falls
  by about half, from 1.00 to 0.52.
- The capital stock falls by much more than 35 percent. Desired capital is proportional to
  $uc^{-3/2}$, so it falls by the factor $(1-\tau)^{3/2} = 0.65^{1.5} \approx 0.52$.
- A higher interest rate moves *both* user-cost lines up, and the tax multiplies every change in
  $R$ by $1/(1-\tau)$.

### Key numbers: from desired capital to investment

After the tax, the firm has more capital than it wants. Equation (17.8) says it can simply stop
investing and let depreciation do the work. How many years does that take?

> **Python note.** A `while` loop repeats its indented lines as long as its condition is true. Here
> each pass is one year in which the capital stock shrinks by the depreciation rate, and `years += 1`
> adds one to the counter.

In [15]:
K_initial = desired_capital(user_cost(0.02, 0.08))
K_target = desired_capital(user_cost(0.02, 0.08, tau=0.35))
K, years = K_initial, 0
while K > K_target:
    K = (1 - 0.08) * K                  # no new investment: equation (17.8) with I = 0
    years += 1
print(f"Capital falls from {K_initial:.2f} to {K_target:.2f}: {years} years with no investment at all")

Capital falls from 1.00 to 0.52: 8 years with no investment at all


In the long run the tax also lowers the **investment rate**. Equation (17.13) says that, for a given
growth rate of capital $g_K$, the investment rate is inversely proportional to the user cost. We use
$g_K = 2$ percent, an illustrative long-run growth rate:

In [16]:
s_before = investment_rate(user_cost(0.02, 0.08))
s_after = investment_rate(user_cost(0.02, 0.08, tau=0.35))
print(f"Investment rate I/Y without the tax: {s_before:.1%}")
print(f"Investment rate I/Y with a 35% tax:  {s_after:.1%}  (a fall by the factor 1 - tau = {s_after / s_before:.2f})")

Investment rate I/Y without the tax: 33.3%
Investment rate I/Y with a 35% tax:  21.7%  (a fall by the factor 1 - tau = 0.65)


### Key numbers: Table 17.1, corporate taxes across countries

Table 17.1 of the textbook lists corporate tax rates in 33 countries and the user cost they imply. The
textbook assumes a user cost of 10 percent in the United States, where the tax rate is 21 percent. The
user cost before taxes is then $10 \times (1-0.21) = 7.9$ percent everywhere, and each country's user
cost is $7.9/(1-\tau)$, as in equation (17.6). We load the OECD tax rates for every year since 2000,
and the table as printed in the textbook.

> **Python note.** `pd.read_csv` reads a data file into a *DataFrame*, a table with named columns, and
> `.head()` shows its first rows. `taxes[taxes.year == year]` keeps only the rows of one year, and
> `.copy()` makes it a new table to which we can add a column. `book.merge(table, on="country")` joins
> two tables by matching the country names, like a lookup in a spreadsheet.

In [17]:
taxes = pd.read_csv(DATA_URL + "corporate_tax.csv")
book = pd.read_csv(DATA_URL + "table17-1-book.csv")
taxes.head()

,country,iso3,year,cit_rate
0,Brazil,BRA,2000,37.0
1,Brazil,BRA,2001,34.0
2,Brazil,BRA,2002,34.0
3,Brazil,BRA,2003,34.0
4,Brazil,BRA,2004,34.0


Each row of `taxes` is one country in one year; `cit_rate` is the statutory corporate income tax rate
of the central government, in percent.

In [18]:
def tax_table(year=2022):
    """Table 17.1 for one year: tax rates and user costs, with the U.S. user cost fixed at 10 percent."""
    table = taxes[taxes.year == year].copy()
    tau_us = table[table.country == "United States"].cit_rate.iloc[0] / 100
    uc_no_tax = 0.10 * (1 - tau_us)                       # the user cost before taxes
    table["user_cost"] = 100 * uc_no_tax / (1 - table.cit_rate / 100)       # equation (17.6)
    table = book.merge(table[["country", "cit_rate", "user_cost"]], on="country")
    names = {"book_rate": "tax rate, book", "cit_rate": f"tax rate, {year}",
             "book_user_cost": "user cost, book", "user_cost": f"user cost, {year}"}
    return table.rename(columns=names).round(1)


table_2022 = tax_table(2022)
table_2022

,country,"tax rate, book","user cost, book","tax rate, 2022","user cost, 2022"
0,Brazil,34.0,12.0,34.0,12.0
1,Argentina,30.0,11.3,35.0,12.2
2,Democratic Republic of the Congo,30.0,11.3,30.0,11.3
3,Kenya,30.0,11.3,30.0,11.3
4,Nigeria,30.0,11.3,30.0,11.3
5,Australia,30.0,11.3,30.0,11.3
6,Costa Rica,30.0,11.3,30.0,11.3
7,Mexico,30.0,11.3,30.0,11.3
8,Portugal,30.0,11.3,30.0,11.3
9,South Africa,27.0,10.8,28.0,11.0


**Look at the differences.** Three rates of 2022 differ from the textbook. Always ask why before you
use the numbers:

In [19]:
different = table_2022[(table_2022["tax rate, 2022"] - table_2022["tax rate, book"]).abs() > 0.05]
different

,country,"tax rate, book","user cost, book","tax rate, 2022","user cost, 2022"
1,Argentina,30.0,11.3,35.0,12.2
9,South Africa,27.0,10.8,28.0,11.0
30,Chile,10.0,8.8,27.0,10.8


- **Chile:** the textbook's 10 percent is Chile's lower rate for *small businesses*. The statutory rate
  for other companies is 27 percent, the number we keep.
- **South Africa:** the textbook already uses the 27 percent rate that applies from 2023; in 2022 it was 28.
- **Argentina:** since 2021 Argentina taxes profits in brackets of 25, 30 and 35 percent, and the OECD
  reports the top one.

Pick a year in the menu to rebuild the table for that year.

> **Python note.** A `Dropdown` is a menu. `interact` shows the table that `tax_table` returns, and
> builds it again whenever you choose another year.

In [20]:
interact(tax_table, year=Dropdown(options=list(range(2000, 2027)), value=2022, description="year", style=WIDE));

The tax rates range from 0 to 34 percent in the textbook, but the user cost only from 7.9 to 12.0
percent. Taxes matter, but a 34-point difference in tax rates raises the user cost only by about half. The
U.S. rate itself was 35 percent until 2017 (the textbook says 34, the rate on middle
incomes) and 21 percent from 2018:

In [21]:
us = taxes[(taxes.country == "United States") & (taxes.year >= 2015)]
us.set_index("year")[["cit_rate"]].T         # .T turns the table on its side, so the years run across

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
cit_rate,35.0,35.0,35.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0,21.0


**Your turn.** The Tax Cuts and Jobs Act of 2017 cut the U.S. corporate tax rate from 35 to 21 percent.
By what factor did it raise the desired capital stock, according to Figure 2's model? First move the
slider to $\tau = 0.35$ and then to $\tau = 0.21$, then compute the ratio with `desired_capital` and
`user_cost`. Finally, run `tax_table(2025)`: which countries changed their tax rate after 2022?

<details><summary>Hint</summary>

Compute `desired_capital(user_cost(0.02, 0.08, tau=0.21)) / desired_capital(user_cost(0.02, 0.08, tau=0.35))`.
Because $K^{\text{desired}}$ is proportional to $(1-\tau)^{3/2}$, the answer is $(0.79/0.65)^{3/2}$. For the
second part, compare the columns `tax rate, book` and `tax rate, 2025`, as we did above for 2022.

</details>

In [22]:
# Your turn: try it here

## Figure 3: The price-earnings ratio in the stock market

**Goal.** Read the price-earnings ratio of the U.S. stock market with the arbitrage equation for stocks.

Robert Shiller's **price-earnings (P/E) ratio** divides the S&P 500 stock price index by the average
real earnings of its companies over the previous ten years. Equation (17.17) says what the ratio
*should* be: the payout ratio divided by $R-g$. The teal line in the figure is this model ratio. With
the textbook's numbers from the stock example, $R = 5$ percent and $g = 2$ percent, and a payout ratio
of one half, the model ratio is $0.5/0.03 \approx 16.7$, close to the long-run average.

In [23]:
cape = pd.read_csv(DATA_URL + "fig17-3.csv", parse_dates=["date"])
cape.tail(3)

,date,cape,cape_average
1747,2026-08-01,40.9874,17.8044
1748,2026-09-01,40.5809,17.8044
1749,2026-10-01,40.7106,17.8044


Each row is a month. `cape` is the P/E ratio and `cape_average` its average over all months since 1881.
`parse_dates=["date"]` turns the text in the `date` column into real dates.

> **Python note.** `px.area` draws a line and fills the area below it. `cape.iloc[-1]` is the last row
> of the table. `if R > g:` runs the indented lines only when the condition holds; `else:` handles the
> other case. Here it protects us from dividing by zero, because equation (17.16) has no finite price
> when $g \geq R$.

In [24]:
def pe_chart(R=0.05, g=0.02, payout=0.5):
    """Figure 17.3: the P/E ratio, its average, and the model ratio payout / (R - g) of equation (17.17)."""
    last, average = cape.iloc[-1], cape.cape.mean()
    fig = px.area(cape, x="date", y="cape", color_discrete_sequence=["#5aa0d8"],
                  labels={"date": "Year", "cape": "P/E ratio"})
    fig.add_trace(go.Scatter(x=[cape.date.iloc[0], last.date], y=[average, average], mode="lines",
                             name="Average", line=dict(color="black", width=1.5, dash="dash")))
    fig.add_annotation(x="1981-01-01", y=average + 7, text=f"Average = {average:.1f}", showarrow=False, font=dict(size=14))
    model_text = "model: no finite price, because g ≥ R"
    if R > g:
        model = pe_ratio(payout, R, g)
        fig.add_trace(go.Scatter(x=[cape.date.iloc[0], last.date], y=[model, model], mode="lines",
                                 name="Model, equation (17.17)", line=dict(color=TEAL, width=3)))
        model_text = f"model: {payout:.2f} / ({R:.1%} − {g:.2%}) = {model:.1f}"
        if model < 48:
            fig.add_annotation(x=last.date, y=model, text="Model", showarrow=False, xanchor="right",
                               yanchor="bottom", font=dict(color=TEAL, size=14))
    fig.update_layout(title=f"P/E ratio {last.cape:.1f} in {last.date:%b %Y}  |  {model_text}",
                      yaxis=dict(range=[0, 50], tickvals=[0, 10, 20, 30, 40, 50]), showlegend=False)
    return fig

In [25]:
interact(pe_chart,
         R=FloatSlider(value=0.05, min=0.03, max=0.08, step=0.005, description="interest rate R", style=WIDE, readout_format=".3f"),
         g=FloatSlider(value=0.02, min=0.0, max=0.04, step=0.0025, description="dividend growth g", style=WIDE, readout_format=".4f"),
         payout=FloatSlider(value=0.5, min=0.3, max=0.8, step=0.05, description="payout ratio", style=WIDE));

In [26]:
# @title Website version: Figure 3 with a precomputed slider for dividend growth
slider_figure(pe_chart, "g", [0.0, 0.01, 0.02, 0.025, 0.03, 0.035, 0.0375], start=0.02,
              label="dividend growth g")

**What to notice.**

- The ratio is very volatile. It peaked in September 1929, before the crash, and reached its record,
  44.2, in December 1999 at the top of the dot-com boom (the textbook dates this peak "in 2001"; by
  2001 the ratio had already fallen to between 28 and 37). It was below 7 in 1920 and in 1982.
- In 2026 the ratio is above 40 again, close to the 1999 record.
- With the slider, the model line rises slowly at first and then very fast: a P/E ratio of 40 needs
  $R-g = 1.25$ percent, against 3 percent for 16.7. Small changes in expected growth or in the interest
  rate can justify big changes in stock prices. That is why it is hard to tell a bubble from a change
  in the fundamentals.

### Key numbers: averages, peaks and the growth rate the market expects

The textbook prints an average of 17.4. That is the average up to March 2023, the last month in the
textbook's figure. With the data up to today the average is a little higher.

In [27]:
book_sample = cape[cape.date <= "2023-03-01"]
print(f"Average P/E ratio, 1881 to March 2023: {book_sample.cape.mean():.1f}   (textbook: 17.4)")
print(f"Average P/E ratio, 1881 to {cape.date.iloc[-1]:%B %Y}: {cape.cape.mean():.1f}")
for year in [1929, 1999]:
    top = cape.loc[cape[cape.date.dt.year == year].cape.idxmax()]
    print(f"Peak of {year}: {top.cape:.1f} in {top.date:%B %Y}")

Average P/E ratio, 1881 to March 2023: 17.4   (textbook: 17.4)
Average P/E ratio, 1881 to October 2026: 17.8
Peak of 1929: 32.6 in September 1929
Peak of 1999: 44.2 in December 1999


Turn equation (17.17) around: for a given payout ratio, the P/E ratio tells us the value of $R-g$ that the
market's price implies, $R - g = \text{payout}/(P/E)$.

In [28]:
last = cape.iloc[-1]
for name, ratio in [("the long-run average", cape.cape.mean()), (f"{last.date:%B %Y}", last.cape)]:
    print(f"P/E ratio of {ratio:.1f} ({name}): R - g = {0.5 / ratio:.2%} with a payout ratio of 0.5")

P/E ratio of 17.8 (the long-run average): R - g = 2.81% with a payout ratio of 0.5
P/E ratio of 40.7 (October 2026): R - g = 1.23% with a payout ratio of 0.5


Today's prices are consistent with the simple model only if investors expect dividends to grow almost as
fast as the interest rate. Either they do, or prices are above their fundamental value.

**Your turn.** Exercise 8 of the textbook prices a stock that pays a dividend of 10 dollars. Use a `for`
loop and `stock_price` to compute its price for $R = 4$ percent and $g$ = 0, 2, 3 and 3.9 percent. What
happens to the price as $g$ gets close to $R$?

<details><summary>Hint</summary>

Loop over a list of growth rates: `for g in [0.0, 0.02, 0.03, 0.039]:` and print `stock_price(10, 0.04, g)`
inside the loop. With $g = 0$ the price is $10/0.04 = 250$ dollars.

</details>

In [29]:
# Your turn: try it here

## Figure 4: Components of private investment (as a share of GDP)

**Goal.** See which parts of investment move most in recessions.

In the national accounts, private investment has three parts:

- **nonresidential fixed investment**: equipment and structures bought by businesses, and intellectual
  property products such as research and software;
- **residential fixed investment**: new housing;
- **inventory investment**: the change in the stock of goods that firms have produced but not yet sold.

In [30]:
invest = pd.read_csv(DATA_URL + "fig17-4.csv")
recessions = pd.read_csv(DATA_URL + "recessions.csv", parse_dates=["peak", "trough"])
quarterly = pd.read_csv(DATA_URL + "investment_quarterly.csv", parse_dates=["date"])
invest.head()

,year,total,nonres_fixed,residential,inventories
0,1975,15.2681,11.6788,3.9606,-0.3713
1,1976,17.2532,11.7084,4.6312,0.9136
2,1977,19.0512,12.4439,5.5350,1.0723
3,1978,20.3426,13.3771,5.8688,1.0967
4,1979,20.5401,14.2286,5.6270,0.6845


Each row of `invest` is a year, and each column a share of GDP in percent; `total` is the sum of the
three parts. `recessions` lists the U.S. recessions dated by the NBER, and `quarterly` has the total and
residential shares for every quarter.

To draw all the parts with one `px.line` call, Plotly Express wants a **long** table: one row per year
and component. We first give the columns readable names, which then appear in the legend.

> **Python note.** `.rename(columns=...)` renames columns with a dictionary. `.melt(id_vars="year")`
> turns a wide table into a long one: the column names go into a new column (`component`) and the
> values into another (`share`). `pd.to_datetime(...)` turns the year into a date (1 January), so that
> the years line up with the recession dates.

In [31]:
NAMES = {"total": "Total", "nonres_fixed": "Nonresidential fixed", "residential": "Residential",
         "inventories": "Inventory"}
COLORS = {"Total": BLUE, "Nonresidential fixed": GREEN, "Residential": TEAL, "Inventory": "#5aa0d8"}
invest = invest.rename(columns=NAMES)
annual_long = invest.melt(id_vars="year", var_name="component", value_name="share")
annual_long["date"] = pd.to_datetime(annual_long.year.astype(str))

recent = quarterly[quarterly.date >= "1975-01-01"].rename(columns=NAMES)
quarterly_long = recent.melt(id_vars="date", var_name="component", value_name="share")
low = recent.loc[recent[recent.date.dt.year == 2009].Total.idxmin()]      # the lowest quarter of 2009
annual_long.head()

,year,component,share,date
0,1975,Total,15.2681,1975-01-01
1,1976,Total,17.2532,1976-01-01
2,1977,Total,19.0512,1977-01-01
3,1978,Total,20.3426,1978-01-01
4,1979,Total,20.5401,1979-01-01


`shade_recessions` shades the recessions. It adds one shaded rectangle (`add_vrect`) from the peak to the
trough of each recession.

> **Python note.** `zip(a, b)` hands over one pair of values at a time, here the peak and the trough of
> each recession.

In [32]:
def shade_recessions(fig, start):
    """Shade every NBER recession that ends after the date start."""
    for peak, trough in zip(recessions.peak, recessions.trough):
        if trough >= pd.Timestamp(start):
            fig.add_vrect(x0=peak, x1=trough, fillcolor=BLUE, opacity=0.12, line_width=0)
    return fig


def components_chart(frequency="annual"):
    """Figure 17.4: investment and its components, percent of GDP, with recessions shaded."""
    title = "Investment and its components: all of them fall in recessions"
    data = annual_long
    if frequency == "quarterly":
        title = f"Quarterly data: total investment fell to {low.Total:.1f}% of GDP in {low.date.year}Q{low.date.quarter}"
        data = quarterly_long
    fig = px.line(data, x="date", y="share", color="component", color_discrete_map=COLORS, title=title,
                  labels={"date": "Year", "share": "Percent of GDP", "component": "Component"})
    shade_recessions(fig, "1975-01-01")
    fig.add_hline(y=0, line_color="black", line_width=1)
    fig.update_layout(xaxis_range=["1975-01-01", "2026-12-31"], yaxis=dict(range=[-5, 25], tickvals=[0, 5, 10, 15, 20, 25]),
                      legend=dict(orientation="h", x=0.01, y=0.99, title=None))
    return fig

> **Python note.** A `ToggleButtons` widget shows one button per option; the function receives the
> option you click.

In [33]:
interact(components_chart,
         frequency=ToggleButtons(options=["annual", "quarterly"], value="annual", description="data"));

In [34]:
# @title Website version: Figure 4 with a precomputed slider for annual or quarterly data
slider_figure(components_chart, "frequency", ["annual", "quarterly"], start="annual", label="data")

**What to notice.**

- Total investment averaged 17.8 percent of GDP from 1975 to 2025, but it swings a lot: from 19.6 percent
  in 2006 to 13.3 percent in 2009, the lowest value in the figure.
- Residential investment peaked at 6.6 percent of GDP in 2005 and fell to 2.4 percent in 2011, less than
  half: the housing bust of the Great Recession.
- Inventory investment is small but **procyclical**: it was negative in the recession years 1975, 1980,
  1982, 1991, 2001, 2008, 2009 and 2020, and in 1983, just after a recession. Firms run down their
  inventories when sales fall.
- Click a name in the legend to hide or show that component. Click **quarterly**: the textbook's
  "less than 13 percent of GDP in 2009" is a quarterly number; the annual value is 13.3.

### Key numbers: which part of investment falls in a recession?

For four recessions we compute how much each part of investment changed, in percentage points of GDP,
from the year before the recession to its worst year.

In [35]:
table = invest.set_index("year")
rows = []
for start, end in [(1979, 1982), (2000, 2002), (2006, 2009), (2019, 2020)]:
    change = table.loc[end] - table.loc[start]
    change["episode"] = f"{start} to {end}"
    rows.append(change)
pd.DataFrame(rows).set_index("episode").round(2)

,Total,Nonresidential fixed,Residential,Inventory
episode,,,,
1979 to 1982,-3.16,0.28,-2.31,-1.13
2000 to 2002,-2.22,-2.24,0.37,-0.35
2006 to 2009,-6.22,-1.31,-3.37,-1.54
2019 to 2020,-0.47,-0.46,0.47,-0.48


Every recession is different. Housing collapsed in the early 1980s, when interest rates were very high,
and again in 2006–2009, when it explains more than half of the fall of 6.2 points. The dot-com bust of
2000–2002 hit business investment, while housing even grew. In 2020 the recession lasted only two months,
so annual data hide most of it.

### Key numbers: house prices and leverage

The arbitrage equation also prices a house. An investor can keep the down payment in the bank, or buy a
condominium with a mortgage, rent it out for a year and sell it. Equating the two returns gives

$$p^{\text{house}} = \frac{\text{rent}}{\tau\bar{x}R + (1-\tau)R + \bar{d} - \Delta p^{\text{house}}/p^{\text{house}}} \qquad (17.20)$$

where $\bar{x}$ is the down payment as a share of the price and $\tau$ the tax rate at which mortgage
interest can be deducted. We use illustrative numbers: a rent of 6,000 dollars a year, $R = 4$ percent,
$\bar{d} = 2$ percent, $\tau = 0.3$ and expected price growth of 2 percent.

In [36]:
def house_price(rent, R, d, gp, tau, x):
    """Price of a house: rent / (tau x R + (1 - tau) R + d - gp)   (equation 17.20; gp is the expected price growth)."""
    return rent / (tau * x * R + (1 - tau) * R + d - gp)


base = house_price(6000, 0.04, 0.02, 0.02, 0.3, 0.2)
print(f"Baseline price (20% down payment, 2% price growth): {base:,.0f} dollars")
print(f"Down payment falls to 10%:          {house_price(6000, 0.04, 0.02, 0.02, 0.3, 0.1) / base - 1:+.1%}")
print(f"Expected price growth rises to 3%:  {house_price(6000, 0.04, 0.02, 0.03, 0.3, 0.2) / base - 1:+.1%}")

Baseline price (20% down payment, 2% price growth): 197,368 dollars
Down payment falls to 10%:          +4.1%
Expected price growth rises to 3%:  +49.0%


With these numbers, cheaper credit (a smaller down payment) raises the price by about 4 percent, but a
belief that prices will grow 1 point faster raises it by about half. Expected capital gains can feed on
themselves, which is how a housing bubble grows.

**Your turn.** The textbook says private investment "has averaged about 17 percent of GDP in recent
decades". Compute the average of the `Total` column of `invest` since 2000. Then compute the share of
nonresidential fixed investment in total investment in 2025.

<details><summary>Hint</summary>

Filter the years with `invest[invest.year >= 2000]` and use `.Total.mean()`. For the share, take the 2025
row, `invest[invest.year == 2025]`, and divide its `"Nonresidential fixed"` column by its `Total` column.

</details>

In [37]:
# Your turn: try it here

## Key numbers

This table collects the main numbers of the chapter. Each one is computed with the functions and data
above, next to the value in the textbook.

In [38]:
uc0, uc1 = user_cost(0.02, 0.08), user_cost(0.02, 0.08, tau=0.35)
rows = [
    ["User cost of capital, R = 2%, d = 8% (percent)", 100 * uc0, "10"],
    ["User cost with a 35% corporate tax (percent)", 100 * uc1, "about 15.3"],
    ["Desired capital with the tax / without", desired_capital(uc1) / desired_capital(uc0), "smaller"],
    ["Investment rate with the tax / without", investment_rate(uc1) / investment_rate(uc0), "lower"],
    ["User cost, Brazil, 2022 (percent)", table_2022.set_index("country").loc["Brazil", "user cost, 2022"], "12.0"],
    ["User cost, Cayman Islands, 2022 (percent)", table_2022.set_index("country").loc["Cayman Islands", "user cost, 2022"], "7.9"],
    ["Stock price, 10 dollar dividend, R = 5% (dollars)", stock_price(10, 0.05), "200"],
    ["Stock price, same with g = 2% (dollars)", stock_price(10, 0.05, 0.02), "about 333"],
    ["Average P/E ratio, 1881 to March 2023", book_sample.cape.mean(), "17.4"],
    ["Average P/E ratio, 1881 to today", cape.cape.mean(), "—"],
    ["Total investment, 2006 (percent of GDP)", table.loc[2006, "Total"], "nearly 20"],
    ["Total investment, 2009, annual (percent of GDP)", table.loc[2009, "Total"], "less than 13"],
    ["Total investment, lowest quarter of 2009 (percent of GDP)", low.Total, "less than 13"],
    ["Residential investment, 2005 peak (percent of GDP)", table.Residential.max(), "more than 6"],
    ["Residential investment, 2011 trough (percent of GDP)", table.Residential.min(), "less than half"],
]
pd.DataFrame(rows, columns=["Quantity", "Notebook", "Textbook"]).round(2)

,Quantity,Notebook,Textbook
0,"User cost of capital, R = 2%, d = 8% (percent)",10.00,10
1,User cost with a 35% corporate tax (percent),15.38,about 15.3
2,Desired capital with the tax / without,0.52,smaller
3,Investment rate with the tax / without,0.65,lower
4,"User cost, Brazil, 2022 (percent)",12.00,12.0
5,"User cost, Cayman Islands, 2022 (percent)",7.90,7.9
6,"Stock price, 10 dollar dividend, R = 5% (dollars)",200.00,200
7,"Stock price, same with g = 2% (dollars)",333.33,about 333
8,"Average P/E ratio, 1881 to March 2023",17.36,17.4
9,"Average P/E ratio, 1881 to today",17.80,—


## Summary

- The **arbitrage equation** says that two equally safe investments must earn the same return. Applied
  to physical capital, it says that firms invest until the marginal product of capital falls to the
  **user cost of capital**, $R + \bar{d} - \Delta p_k/p_k$.
- A corporate income tax divides the user cost by $1-\tau$. With diminishing returns, a 35 percent tax
  cuts the desired capital stock by about half and the investment rate by 35 percent.
- Applied to stocks, the same equation gives $p_s = \text{dividend}/(R-g)$: a stock is worth the present
  value of its dividends. The P/E ratio then depends on $R-g$, which makes it very sensitive to expected
  growth and hard to use for spotting bubbles.
- In the data, total investment and especially housing and inventories fall sharply in recessions.
- The next chapter turns to **the government and the macroeconomy**: how governments spend, tax and
  borrow, and what their budget constraint implies for debt and deficits.